# DATA 298B Final Gemma Validation

This notebook runs Gemma only on 10 unseen cases. It does not retrain models or rerun the four-model benchmark.

In [ ]:
# 1. Fresh-runtime setup and Drive mount
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/Graph-Based-Fraud-Intelligence-Platform'
DRIVE_ROOT = '/content/drive/MyDrive/DATA298B'
VALIDATION_ROOT = f'{DRIVE_ROOT}/final_validation_gemma'
!mkdir -p $VALIDATION_ROOT

In [ ]:
# 2. Pull the backend branch
# Replace YOUR_REPOSITORY_URL with the GitHub repository URL.
%cd /content
REPOSITORY_URL = 'YOUR_REPOSITORY_URL'
if not os.path.exists(REPO_DIR):
    !git clone -b feature/298b-backend $REPOSITORY_URL $REPO_DIR
%cd $REPO_DIR
print('Repository:', os.getcwd())

## Upload the two latest local files if they are not committed to GitHub

Upload `build_integrated_298b_benchmark.py` and the updated `schemas.py` when prompted. If both changes are already on the branch, skip the next cell.

In [ ]:
# 3. Optional: upload the latest local generator and schema
from google.colab import files
uploaded = files.upload()
if 'build_integrated_298b_benchmark.py' in uploaded:
    !mv build_integrated_298b_benchmark.py scripts/build_integrated_298b_benchmark.py
if 'schemas.py' in uploaded:
    !mv schemas.py src/aml/schemas.py
print('Upload step complete')

In [ ]:
# 4. Build 5 suspicious + 5 legitimate cases, excluding the old 50-case benchmark
import subprocess

subprocess.run([
    'python', 'scripts/build_integrated_298b_benchmark.py',
    '--parquet', f'{DRIVE_ROOT}/input/test_graph_enriched.parquet',
    '--patterns', f'{DRIVE_ROOT}/input/HI-Medium_Patterns.txt',
    '--xgboost-model', 'data/models/xgboost_graph_enhanced.json',
    '--exclude-jsonl', f'{DRIVE_ROOT}/heldout/integrated_cases.jsonl',
    '--output', f'{VALIDATION_ROOT}/validation_cases.jsonl',
    '--case-prefix', 'validation_',
    '--suspicious-cases', '5',
    '--legitimate-cases', '5',
], check=True)

In [ ]:
# 5. Load the ten validation cases
import json
from pathlib import Path

VALIDATION_DATASET = Path(f'{VALIDATION_ROOT}/validation_cases.jsonl')
records = [json.loads(line) for line in VALIDATION_DATASET.read_text().splitlines() if line.strip()]
print('Validation cases:', len(records))
print('Case IDs:', [record['case_id'] for record in records])
assert len(records) == 10

In [ ]:
# 6. Load the Gemma adapter and define deterministic retrieval
import gc, re, time, uuid
import yaml
import torch
from src.aml.backend import TransformersBackend
from src.aml.schemas import InvestigationCase, InvestigationReport
from src.aml.tools import execute_tool
from src.aml.report_guardrails import apply_report_guardrails

adapter_candidates = list(Path(DRIVE_ROOT, 'adapters').glob('*gemma*'))
adapter_candidates = [p for p in adapter_candidates if (p / 'adapter_config.json').exists()]
if not adapter_candidates:
    raise FileNotFoundError('No Gemma adapter found under DATA298B/adapters')
ADAPTER_DIR = adapter_candidates[0]
registry = yaml.safe_load(Path('configs/models/registry.yaml').read_text())
spec = registry['gemma']
backend = TransformersBackend(spec['model_id'], revision=spec['revision'], adapter_path=str(ADAPTER_DIR), model_kwargs={'load_in_4bit': True, 'device_map': 'auto'})
print('Loaded adapter:', ADAPTER_DIR)

def parse_json_response(text):
    match = re.search(r'\{.*\}', text, re.S)
    if not match:
        raise ValueError('No JSON object found in model response')
    return json.loads(match.group(0))

SYSTEM = ('You are an AML investigator. Return JSON only with exactly these keys: '
          'decision, pattern, risk_level, confidence, evidence, recommended_actions, summary. '
          'Use only facts in the case context and retrieved tool results. Do not invent facts. '
          'A SUSPICIOUS decision requires at least one concrete supported fact. '
          'If the evidence does not support a fraud pattern, use NONE or EMERGING_UNKNOWN.')

In [ ]:
# 7. Run Gemma on the ten unseen cases; this cell resumes completed cases
TRACE_DIR = Path(VALIDATION_ROOT) / 'traces'
TRACE_DIR.mkdir(parents=True, exist_ok=True)
TRACE_JSONL = Path(VALIDATION_ROOT) / 'gemma_validation_traces.jsonl'
completed = set()
if TRACE_JSONL.exists():
    completed = {json.loads(line)['case_id'] for line in TRACE_JSONL.read_text().splitlines() if line.strip()}

with TRACE_JSONL.open('a') as output:
    for index, record in enumerate(records, 1):
        case_id = record['case_id']
        if case_id in completed:
            print(f'{index}/10 {case_id} already complete', flush=True)
            continue
        case = InvestigationCase(case_id=case_id, context=record['context'], ground_truth=record['ground_truth'], split='test')
        retrieved = []
        tool_calls = []
        for tool_name in ('transaction_lookup', 'graph_lookup', 'community_lookup'):
            call = {'name': tool_name, 'arguments': {}, 'success': False}
            try:
                result = execute_tool(tool_name, case.agent_context(), {})
                call['success'] = True
                call['result'] = result
                retrieved.append(result)
            except Exception as exc:
                call['error'] = repr(exc)
            tool_calls.append(call)
        prompt = {'case_context': case.agent_context(), 'retrieved_tool_results': retrieved, 'instruction': 'Return the final grounded JSON report now.'}
        started = time.perf_counter()
        response = backend.generate([{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': json.dumps(prompt, default=str)}], max_tokens=400, temperature=0.0)
        raw = response.get('text', '')
        try:
            obj = parse_json_response(raw)
            report = InvestigationReport(decision=obj.get('decision', 'SUSPICIOUS'), pattern=obj.get('pattern', 'EMERGING_UNKNOWN'), risk_level=obj.get('risk_level', 'MEDIUM'), confidence=float(obj.get('confidence', 0.0)), evidence=obj.get('evidence', []), recommended_actions=obj.get('recommended_actions', []), summary=obj.get('summary', ''))
        except Exception as exc:
            report = InvestigationReport(summary='Invalid model output')
            raw = raw + '\nPARSER_ERROR: ' + repr(exc)
        guarded_report = apply_report_guardrails(report.__dict__, case.agent_context())
        trace = {'trace_id': uuid.uuid4().hex, 'case_id': case_id, 'model_family': 'gemma', 'variant': 'final_unseen_validation_guarded', 'seed': 42, 'tool_budget': 3, 'tool_calls': tool_calls, 'tool_budget_hit': False, 'context': case.agent_context(), 'ground_truth': case.ground_truth, 'report': guarded_report, 'raw_response': raw, 'latency_seconds': time.perf_counter() - started, 'prompt_tokens': response.get('prompt_tokens', 0), 'output_tokens': response.get('output_tokens', 0)}
        output.write(json.dumps(trace, default=str) + '\n')
        output.flush()
        print(f'{index}/10 {case_id} complete', flush=True)

In [ ]:
# 8. Save final validation metrics and verify artifacts
from src.evaluation.benchmark_runner import run as evaluate_traces
metrics = evaluate_traces(TRACE_JSONL, Path(VALIDATION_ROOT) / 'metrics')
(Path(VALIDATION_ROOT) / 'metrics_summary.json').write_text(json.dumps(metrics, indent=2, default=str))
print(json.dumps(metrics, indent=2, default=str))
print('Traces:', TRACE_JSONL)
print('Metrics:', Path(VALIDATION_ROOT) / 'metrics_summary.json')

In [ ]:
# 9. Optional: run only after all files above are saved
import gc
del backend
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Validation complete. Disconnecting runtime...')
from google.colab import runtime
runtime.unassign()